# Data Mining Assignment
## Apache Spark Data Lake Preprocessing in Google Colab

**Student Name:** Srivastav  
**Roll Number:** LIT2023037  
**Dataset:** sales.csv  
**Environment:** Google Colab + PySpark

---

### Assignment Objective
Using the provided `sales.csv` dataset, create a simple Data Lake architecture in Google Colab consisting of **Bronze** and **Silver** layers:
1. **Bronze Layer:** Store the raw dataset without modification.
2. **Silver Layer:** Perform data inspection and basic preprocessing with Apache Spark, handle discrepancies (duplicates, missing values, text formatting, numerical boundaries, dates), and save the preprocessed data.

In [1]:
!pip install -q pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DateType
import os
import shutil
import urllib.request

## 1. Create Spark Session

Initialize an Apache Spark session configured for data processing and date compatibility.

In [3]:
spark = SparkSession.builder \
    .appName("SalesDataLakePreprocessing_LIT2023037") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.4

## 2. Create Data Lake Structure

Create the required folder hierarchy in the Colab runtime environment:
```
data_lake/
|-- bronze/
|   `-- sales.csv
`-- silver/
    `-- preprocessed_sales/
```
The raw file in the Bronze layer remains completely untouched and unmodified.

In [4]:
# Create Data Lake directories
os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

# Populate Bronze layer with raw sales.csv
bronze_file = "data_lake/bronze/sales.csv"
if os.path.exists("dataset/sales.csv"):
    shutil.copy2("dataset/sales.csv", bronze_file)
elif os.path.exists("sales.csv"):
    shutil.copy2("sales.csv", bronze_file)
else:
    raw_url = "https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv"
    urllib.request.urlretrieve(raw_url, bronze_file)

print("Data Lake directories and Bronze dataset ready.")
print("Bronze file exists:", os.path.exists(bronze_file))

Data Lake directories and Bronze dataset ready.
Bronze file exists: True

## 3. Load and Inspect Bronze Layer with Spark

Read `sales.csv` from the Bronze layer using PySpark, inspect the schema, display initial rows, and check the total record count.

In [5]:
bronze_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(bronze_file)

bronze_df.show(5)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|  100001|      C0256|Keerthi Mishra|        Saree|Clothing|       2|      6710|              20|   Credit Card|    Mumbai|   Maharashtra|2026-01-01|   Delivered|
|  100002|      C0042|   Arjun Singh|      T-Shirt|Clothing|       6|       680|               5|    Debit Card|   Chennai|    Tamil Nadu|2025-08-03|   Delivered|
|  100003|      C0348|   Keerthi Rao|       Hoodie|Clothing|       6|      3080|              12|           UPI|      Pune|   Maharashtra|2025-05-06|     Shipped|
|  100004|      C0051|

In [6]:
bronze_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

In [7]:
initial_count = bronze_df.count()
print("Total records in Bronze layer:", initial_count)

Total records in Bronze layer: 1000

### Column Names and Data Types

In [8]:
print("Columns:")
for column, dtype in bronze_df.dtypes:
    print(f"  - {column:18} : {dtype}")

Columns:
  - order_id           : int
  - customer_id        : string
  - customer_name      : string
  - product            : string
  - category           : string
  - quantity           : int
  - unit_price         : int
  - discount_percent   : int
  - payment_method     : string
  - city               : string
  - state              : string
  - order_date         : string
  - order_status       : string

## 4. Identify Data Discrepancies Using Spark Operations

Inspect the Bronze layer to detect:
1. Exact duplicate rows
2. Missing / Null values across all columns
3. Empty string values in text fields
4. Text formatting & capitalization inconsistencies
5. Numerical anomalies (quantity <= 0, unit price <= 0, invalid discount percentages)
6. Inconsistent / invalid date values

In [9]:
# 4.1 Exact Duplicate Rows
distinct_count = bronze_df.dropDuplicates().count()
duplicate_count = initial_count - distinct_count

print("Total records   :", initial_count)
print("Distinct records:", distinct_count)
print("Duplicate rows  :", duplicate_count)

Total records   : 1000
Distinct records: 990
Duplicate rows  : 10

In [10]:
# Display the duplicated records
duplicate_rows = bronze_df.groupBy(bronze_df.columns) \
    .count() \
    .filter(F.col("count") > 1)

duplicate_rows.show(truncate=False)

+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name   |product          |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100692  |C0141      |Varun Agarwal   |Electric Kettle  |Home Appliances|4       |920       |0               |Net Banking     |Delhi     |Delhi         |2026-08-18|Shipped     |2    |
|100145  |C0007      |Sai Mishra      |Face Wash        |Beauty         |7       |350       |30              |Cash on Delivery|Bengaluru |Karnataka     |2025-08-28|Delivered   |2    |
|100297  |C0149      |Manish Pillai   |Dry Fruits Pack  |Grocery        |7      

In [11]:
# 4.2 Missing / Null Values per Column
null_counts = bronze_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in bronze_df.columns
])

null_counts.show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

In [12]:
# 4.3 Check for Empty or Whitespace-only Strings
text_columns = [
    field.name
    for field in bronze_df.schema.fields
    if field.dataType.simpleString() == "string"
]

empty_counts = bronze_df.select([
    F.sum(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in text_columns
])

empty_counts.show()

+-----------+-------------+-------+--------+--------------+----+-----+----------+------------+
|customer_id|customer_name|product|category|payment_method|city|state|order_date|order_status|
+-----------+-------------+-------+--------+--------------+----+-----+----------+------------+
|          0|           10|      0|       0|             8|   8|    0|         0|           0|
+-----------+-------------+-------+--------+--------------+----+-----+----------+------------+

In [13]:
# 4.4 Text-Format & Capitalization Inconsistencies
print("Distinct Category values in Bronze layer:")
bronze_df.select("category").distinct().show()

print("Distinct City values with extra whitespace or mixed casing in Bronze layer:")
bronze_df.select("city").distinct().filter(
    (F.col("city") != F.trim(F.col("city"))) |
    (F.col("city") == F.upper(F.col("city")))
).show()

Distinct Category values in Bronze layer:
+---------------+
|category       |
+---------------+
|sports         |
|beauty         |
|ELECTRONICS    |
|Home Appliances|
|Books          |
|Grocery        |
|Sports         |
|Furniture      |
|CLOTHING       |
|Clothing       |
|SPORTS         |
|Electronics    |
|Beauty         |
+---------------+

Distinct City values with extra whitespace or mixed casing in Bronze layer:
+-----------+
|city       |
+-----------+
|    Mumbai |
|HYDERABAD  |
|DELHI      |
| Hyderabad |
+-----------+

In [14]:
# 4.5 Invalid Numerical Values
invalid_qty = bronze_df.filter(F.col("quantity") <= 0)
invalid_price = bronze_df.filter(F.col("unit_price") <= 0)
invalid_disc = bronze_df.filter((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100))

print(f"Invalid Quantity records (<= 0)        : {invalid_qty.count()}")
print(f"Invalid Unit Price records (<= 0)      : {invalid_price.count()}")
print(f"Invalid Discount records (< 0 or > 100): {invalid_disc.count()}")

Invalid Quantity records (<= 0)        : 12
Invalid Unit Price records (<= 0)      : 8
Invalid Discount records (< 0 or > 100): 8

In [15]:
# 4.6 Inconsistent and Invalid Dates
date_check_df = bronze_df.withColumn(
    "parsed_date",
    F.try_to_timestamp(F.col("order_date"), F.lit("yyyy-MM-dd")).cast("date")
)

invalid_dates = date_check_df.filter(F.col("parsed_date").isNull())
print("Total inconsistent / invalid date records:", invalid_dates.count())
invalid_dates.select("order_id", "customer_id", "order_date").show(truncate=False)

Total inconsistent / invalid date records: 6
+--------+-----------+----------+
|order_id|customer_id|order_date|
+--------+-----------+----------+
|100097  |C0244      |31/02/2026|
|100358  |C0132      |2025/07/14|
|100448  |C0270      |not-a-date|
|100634  |C0262      |2025-02-30|
|100637  |C0020      |2026-13-05|
|100969  |C0277      |09-15-2026|
+--------+-----------+----------+

## 5. Basic Preprocessing - Silver Layer

Perform the following preprocessing operations to prepare the Silver layer:
1. **Remove exact duplicate rows** (`dropDuplicates()`).
2. **Trim whitespace** from text fields.
3. **Standardize capitalization** for text fields (`category`, `city`).
4. **Handle missing values**: Impute missing `customer_name`, `payment_method`, and `city` with `'Unknown'` to preserve useful records.
5. **Numerical validation**: Convert non-positive `quantity` and `unit_price`, and out-of-bound `discount_percent` (< 0 or > 100) to `NULL`.
6. **Date standardization**: Safely convert valid dates to the unified `yyyy-MM-dd` format; convert invalid dates to `NULL`.
7. **Preserve useful records**: Retain all valid business transactions rather than dropping entire rows.

In [16]:
# Step 1: Remove exact duplicate rows
silver_df = bronze_df.dropDuplicates()

print("Rows before duplicate removal:", bronze_df.count())
print("Rows after duplicate removal :", silver_df.count())

Rows before duplicate removal: 1000
Rows after duplicate removal : 990

In [17]:
# Step 2: Trim extra spaces from text columns
for column in text_columns:
    silver_df = silver_df.withColumn(column, F.trim(F.col(column)))

In [18]:
# Step 3: Standardize capitalization inconsistencies
silver_df = silver_df.withColumn("category", F.initcap(F.col("category")))
silver_df = silver_df.withColumn("city", F.initcap(F.col("city")))

In [19]:
# Step 4: Handle missing values by imputing 'Unknown' to preserve records
for col_name in ["customer_name", "payment_method", "city"]:
    silver_df = silver_df.withColumn(
        col_name,
        F.when(
            F.col(col_name).isNull() | (F.trim(F.col(col_name)) == ""),
            F.lit("Unknown")
        ).otherwise(F.col(col_name))
    )

In [20]:
# Step 5: Apply basic numerical validation (setting invalid values to NULL)
# Validate quantity (> 0)
silver_df = silver_df.withColumn(
    "quantity",
    F.when(F.col("quantity") > 0, F.col("quantity")).otherwise(F.lit(None))
)

# Validate unit_price (> 0)
silver_df = silver_df.withColumn(
    "unit_price",
    F.when(F.col("unit_price") > 0, F.col("unit_price")).otherwise(F.lit(None))
)

# Validate discount_percent (0 to 100)
silver_df = silver_df.withColumn(
    "discount_percent",
    F.when(
        (F.col("discount_percent") >= 0) & (F.col("discount_percent") <= 100),
        F.col("discount_percent")
    ).otherwise(F.lit(None))
)

In [21]:
# Step 6: Convert order_date safely to DateType (yyyy-MM-dd)
# Clearly invalid dates safely evaluate to NULL without failing
silver_df = silver_df.withColumn(
    "order_date",
    F.try_to_timestamp(F.col("order_date"), F.lit("yyyy-MM-dd")).cast("date")
)

In [22]:
silver_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

In [23]:
final_count = silver_df.count()

print("Row count before preprocessing:", initial_count)
print("Row count after preprocessing :", final_count)
print("Rows removed (exact duplicates):", initial_count - final_count)

Row count before preprocessing: 1000
Row count after preprocessing : 990
Rows removed (exact duplicates): 10

In [24]:
silver_df.show(10, truncate=False)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books      

In [25]:
# Save Silver data in CSV format
silver_output_dir = "data_lake/silver/preprocessed_sales"
final_silver_file = "data_lake/silver/sales_silver.csv"

if os.path.exists(silver_output_dir):
    shutil.rmtree(silver_output_dir)

silver_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_output_dir)

# Locate generated part file and save as sales_silver.csv
part_files = [os.path.join(silver_output_dir, f) for f in os.listdir(silver_output_dir) if f.startswith("part-") and f.endswith(".csv")]
if part_files:
    shutil.copy2(part_files[0], final_silver_file)
    print(f"Silver CSV saved successfully to {final_silver_file}")

Silver CSV saved successfully to data_lake/silver/sales_silver.csv

## 6. Verification

Read the saved Silver dataset back using Spark to verify schema, row count, duplicate absence, and numerical validity.

In [26]:
silver_read_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(final_silver_file)

silver_read_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

In [27]:
silver_read_df.show(10, truncate=False)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books      

In [28]:
silver_read_count = silver_read_df.count()
remaining_duplicates = silver_read_count - silver_read_df.dropDuplicates().count()

invalid_qty_remaining = silver_read_df.filter(F.col("quantity") <= 0).count()
invalid_price_remaining = silver_read_df.filter(F.col("unit_price") <= 0).count()
invalid_disc_remaining = silver_read_df.filter((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)).count()

print(f"Silver record count            : {silver_read_count}")
print(f"Remaining exact duplicates     : {remaining_duplicates}")
print(f"Remaining invalid quantities   : {invalid_qty_remaining}")
print(f"Remaining invalid unit prices  : {invalid_price_remaining}")
print(f"Remaining invalid discounts    : {invalid_disc_remaining}")

Silver record count            : 990
Remaining exact duplicates     : 0
Remaining invalid quantities   : 0
Remaining invalid unit prices  : 0
Remaining invalid discounts    : 0

## 7. Preprocessing Summary

The following preprocessing operations were performed to transform raw Bronze data into curated Silver data:

1. **Bronze Layer Ingestion:** Loaded the raw `sales.csv` using Apache Spark and preserved the original data without modifications.
2. **Data Lake Architecture:** Structured the Colab workspace into `data_lake/bronze` and `data_lake/silver/preprocessed_sales`.
3. **Duplicate Elimination:** Identified 10 exact duplicate rows and removed them using `dropDuplicates()`, reducing total records from 1,000 to 990.
4. **Missing Value Imputation:** Detected missing values in `customer_name` (10), `payment_method` (8), and `city` (8). Replaced missing and empty strings with `'Unknown'` to retain valuable sales records.
5. **Whitespace Normalization:** Trimmed leading and trailing spaces from all string columns using `F.trim()`.
6. **Capitalization Standardization:** Converted text fields (`category`, `city`) to standard Title Case using `F.initcap()`, resolving discrepancies such as `'sports'` $\rightarrow$ `'Sports'`, `'CLOTHING'` $\rightarrow$ `'Clothing'`, and `'DELHI'` $\rightarrow$ `'Delhi'`.
7. **Quantity Validation:** Flagged 12 records with non-positive values ($\le 0$) and set them to `NULL`, preserving the order records.
8. **Unit Price Validation:** Flagged 8 records with non-positive prices ($\le 0$) and set them to `NULL`.
9. **Discount Percent Validation:** Flagged 8 records with out-of-range discount percentages ($<0$ or $>100$) and set them to `NULL`.
10. **Date Parsing and Standardization:** Safely parsed order dates to standard `yyyy-MM-dd` format. Handled 6 non-standard/invalid dates (`31/02/2026`, `not-a-date`, etc.) by safely converting them to `NULL`.
11. **Record Preservation:** Retained all 990 valid transaction records without discarding orders due to individual column discrepancies.
12. **Silver Output Generation:** Exported the final cleaned DataFrame to `data_lake/silver/preprocessed_sales` and created `sales_silver.csv`.
13. **Verification:** Read back the saved Silver CSV via PySpark, confirming schema consistency, zero duplicates, and zero boundary violations.

In [29]:
spark.stop()
print("SparkSession stopped successfully.")

SparkSession stopped successfully.